# Foundation Part Categories — Silver Layer

Reads current part category records from `lego_brickbase.bronze.raw_part_categories` and loads them into the `lego_brickbase.silver.foundation_part_categories` Delta table.

## What this notebook does

1. **Read** — Selects all current, non-deleted records from the bronze `raw_part_categories` table.
2. **Transform** — Renames `id` to `part_category_key`, retains business columns (`id`, `name`, `part_count`, `valid_from`), and derives a `valid_from_date` column used for partitioning.
3. **Write** — Overwrites the Delta table at `/Volumes/lego_brickbase/silver/delta_volume/part_categories`, partitioned by `valid_from_date`.
4. **Register** — Creates the catalog table `lego_brickbase.silver.foundation_part_categories` in Unity Catalog if it does not already exist.

## Imports and Configuration

Import Spark functions needed for the transformation. `BRONZE_TABLE` is the source; `DELTA_TABLE_PATH` is the physical Delta location on the Unity Catalog external volume; `CATALOG_TABLE` is the Unity Catalog table reference.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date

spark = SparkSession.builder.getOrCreate()

BRONZE_TABLE     = "lego_brickbase.bronze.raw_part_categories"
DELTA_TABLE_PATH = "/Volumes/lego_brickbase/silver/delta_volume/part_categories"
CATALOG_TABLE    = "lego_brickbase.silver.foundation_part_categories"

## Read and Transform

Filter the bronze table to current, non-deleted records, then rename `id` to `part_category_key` and derive `valid_from_date` (the date portion of `valid_from`) for use as the Delta partition column.

In [ ]:
df_source = (
    spark.table(BRONZE_TABLE)
    .filter((col("is_current") == True) & (col("is_deleted") == False))
)

df_foundation = (
    df_source
    .select(
        col("id").alias("part_category_key"),
        col("id"),
        col("name"),
        col("part_count"),
        col("valid_from"),
    )
    .withColumn("valid_from_date", to_date(col("valid_from")))
)

df_foundation.printSchema()
df_foundation.display(10, truncate=False)

## Write to Delta Volume and Register Catalog Table

Overwrite the Delta table at the silver volume path, partitioned by `valid_from_date`. Then register it in Unity Catalog as `lego_brickbase.silver.foundation_part_categories` if it does not already exist.

In [ ]:
(
    df_foundation
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("valid_from_date")
    .save(DELTA_TABLE_PATH)
)

row_count = spark.read.format("delta").load(DELTA_TABLE_PATH).count()
print(f"Rows written to Delta table: {row_count:,}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG_TABLE}
    AS SELECT * FROM delta.`{DELTA_TABLE_PATH}`
""")
print(f"Catalog table ready: {CATALOG_TABLE}")